# Parse FASTA Files from MC_fasta Directory

This notebook reads FASTA files from the `B_metaclusters_xml/mc_info_from_xml/MC_fasta/` folder, extracts protein information, and creates a comprehensive DataFrame with columns: `mcid`, `protein`, `range`,`aa_length`, and `aa_seq`.

## Section 1: Import Required Libraries

In [1]:
# Necessary imports 
import pandas as pd
import os
from pathlib import Path
from typing import List, Tuple, Dict

## Section 2: Set Up File Path and Parameters

In [2]:
# Path to the FASTA folder
fasta_folder = Path('/u/mdmc/enyanduk/internship_areasciencepark/Data/dpcfam/dpcfam_b/zenodo_unzipped_folders/B_metaclusters_xml/mc_info_from_xml/MC_fasta/')

# Verify the folder exists
if fasta_folder.exists():
    print(f"Folder found at: {fasta_folder.absolute()}")
else:
    print(f"Folder not found at: {fasta_folder}")

# List all FASTA files in the folder
fasta_files = list(fasta_folder.glob('*.fasta'))
print(f"\nFound {len(fasta_files)} FASTA files")
if fasta_files:
    print("First few files:")
    for f in fasta_files[:5]:
        print(f"  - {f.name}")

Folder found at: /u/mdmc/enyanduk/internship_areasciencepark/Data/dpcfam/dpcfam_b/zenodo_unzipped_folders/B_metaclusters_xml/mc_info_from_xml/MC_fasta

Found 34556 FASTA files
First few files:
  - MC113651.fasta
  - MC195107.fasta
  - MC221075.fasta
  - MC53948.fasta
  - MC322411.fasta


## Section 3: Create Function to Parse FASTA Files

In [3]:
def parse_fasta_file(filepath: Path) -> List[Tuple[str, str, str]]:
    """
    Parse a FASTA file and extract protein, range, and amino acid sequence.
    
    Args:
        filepath: Path to the FASTA file
        
    Returns:
        List of tuples containing (protein_id, range, aa_sequence)
    """
    entries = []
    
    with open(filepath, 'r') as f:
        current_protein = None
        current_range = None
        current_sequence = []
        
        for line in f:
            line = line.strip()
            
            if line.startswith('>'):
                # Save the previous entry if it exists
                if current_protein is not None:
                    aa_sequence = ''.join(current_sequence)
                    entries.append((current_protein, current_range, aa_sequence))
                
                # Parse the header line
                # Format: >Protein/Range
                header = line[1:]  # Remove the '>'
                
                if '/' in header:
                    protein_id, range_str = header.split('/', 1)
                    current_protein = protein_id
                    current_range = range_str
                else:
                    # Handle case where there's no range
                    current_protein = header
                    current_range = ''
                
                current_sequence = []
            else:
                # This is a sequence line
                if line:  # Only add non-empty lines
                    current_sequence.append(line)
        
        # Don't forget to add the last entry
        if current_protein is not None:
            aa_sequence = ''.join(current_sequence)
            entries.append((current_protein, current_range, aa_sequence))
    
    return entries

In [4]:
# Test the function on a sample file
if fasta_files:
    sample_file = fasta_files[0]
    print(f"Testing parser on: {sample_file.name}")
    sample_entries = parse_fasta_file(sample_file)
    print(f"Found {len(sample_entries)} entries")
    if sample_entries:
        print(f"\nFirst entry example:")
        protein, range_val, aa = sample_entries[0]
        print(f"  Protein: {protein}")
        print(f"  Range: {range_val}")
        print(f"  AA (first 100 chars): {aa[:100]}...")
        print(f"  Total AA length: {len(aa)}")

Testing parser on: MC113651.fasta
Found 30 entries

First entry example:
  Protein: A0A0K6GCJ4
  Range: 2-376
  AA (first 100 chars): PTVVSSRALVPIGRRIRNAPEDSDASKAIVLRRKQQGETEDLSKALILRAPDDGRDDRQALVLYRRRGAQEMLRQLVAWHKSSALLPPFRMDELIRIADS...
  Total AA length: 375


## Section 4: Iterate Through FASTA Files in Directory

In [5]:
# Iterate through all FASTA files and collect data
all_data = []

print(f"Processing {len(fasta_files)} FASTA files...\n")

for idx, fasta_file in enumerate(fasta_files, 1):
    # Extract MCID from filename (e.g., MC1.fasta -> MC1)
    mcid = fasta_file.stem  # stem removes the .fasta extension
    
    # Parse the FASTA file
    try:
        entries = parse_fasta_file(fasta_file)
        
        # Add MCID to each entry
        for protein, range_val, aa_sequence in entries:
            all_data.append({
                'MCID': mcid,
                'Protein': protein,
                'Range': range_val,
                'AA': aa_sequence
            })
        
        print(f"[{idx}/{len(fasta_files)}] {fasta_file.name}: {len(entries)} entries processed")
    
    except Exception as e:
        print(f"[{idx}/{len(fasta_files)}] ✗ Error processing {fasta_file.name}: {str(e)}")

print(f"\nTotal entries collected: {len(all_data)}")

Processing 34556 FASTA files...

[1/34556] MC113651.fasta: 30 entries processed
[2/34556] MC195107.fasta: 32 entries processed
[3/34556] MC221075.fasta: 44 entries processed
[4/34556] MC53948.fasta: 45 entries processed
[5/34556] MC322411.fasta: 30 entries processed
[6/34556] MC69046.fasta: 34 entries processed
[7/34556] MC111882.fasta: 40 entries processed
[8/34556] MC142416.fasta: 31 entries processed
[9/34556] MC405129.fasta: 41 entries processed
[10/34556] MC298852.fasta: 36 entries processed
[11/34556] MC16441.fasta: 26 entries processed
[12/34556] MC475976.fasta: 43 entries processed
[13/34556] MC480307.fasta: 25 entries processed
[14/34556] MC118286.fasta: 34 entries processed
[15/34556] MC449689.fasta: 35 entries processed
[16/34556] MC224289.fasta: 28 entries processed
[17/34556] MC119536.fasta: 26 entries processed
[18/34556] MC34470.fasta: 31 entries processed
[19/34556] MC193492.fasta: 42 entries processed
[20/34556] MC91323.fasta: 43 entries processed
[21/34556] MC264220.f

[22/34556] MC59009.fasta: 29 entries processed
[23/34556] MC9618.fasta: 41 entries processed
[24/34556] MC248352.fasta: 41 entries processed
[25/34556] MC87614.fasta: 30 entries processed
[26/34556] MC38651.fasta: 45 entries processed
[27/34556] MC272419.fasta: 43 entries processed
[28/34556] MC51327.fasta: 39 entries processed
[29/34556] MC73756.fasta: 47 entries processed
[30/34556] MC376459.fasta: 37 entries processed
[31/34556] MC211706.fasta: 45 entries processed
[32/34556] MC30509.fasta: 37 entries processed
[33/34556] MC26088.fasta: 31 entries processed
[34/34556] MC116830.fasta: 44 entries processed
[35/34556] MC5509.fasta: 26 entries processed
[36/34556] MC156198.fasta: 38 entries processed
[37/34556] MC20073.fasta: 31 entries processed
[38/34556] MC46619.fasta: 47 entries processed
[39/34556] MC21755.fasta: 26 entries processed
[40/34556] MC6066.fasta: 29 entries processed
[41/34556] MC210038.fasta: 40 entries processed
[42/34556] MC126514.fasta: 29 entries processed
[43/3455

## Section 5: Process and Extract Data

In [6]:
# Add calculated column for sequence length
for entry in all_data:
    entry['Sequence_Length'] = len(entry['AA'])

# Display data statistics
print(f"Total entries: {len(all_data)}")
print(f"Number of unique MCIDs: {len(set(entry['MCID'] for entry in all_data))}")
print(f"Number of unique Proteins: {len(set(entry['Protein'] for entry in all_data))}")
print(f"\nSequence length statistics:")
sequence_lengths = [entry['Sequence_Length'] for entry in all_data]
print(f"  Min: {min(sequence_lengths)}")
print(f"  Max: {max(sequence_lengths)}")
print(f"  Average: {sum(sequence_lengths)/len(sequence_lengths):.2f}")

# Show sample entries
print(f"\nFirst 3 entries:")
for i, entry in enumerate(all_data[:3], 1):
    print(f"  {i}. MCID={entry['MCID']}, Protein={entry['Protein']}, Range={entry['Range']}, "
          f"AA_length={entry['Sequence_Length']}")

Total entries: 1200826
Number of unique MCIDs: 34556
Number of unique Proteins: 1014273

Sequence length statistics:
  Min: 27
  Max: 3545
  Average: 181.18

First 3 entries:
  1. MCID=MC113651, Protein=A0A0K6GCJ4, Range=2-376, AA_length=375
  2. MCID=MC113651, Protein=A8N621, Range=40-384, AA_length=345
  3. MCID=MC113651, Protein=A0A0H2RRJ7, Range=98-399, AA_length=302


## Section 6: Create DataFrame from Extracted Data

In [ ]:
# Create DataFrame with the four required columns
df = pd.DataFrame(all_data)

# Select and reorder columns to match requirements
df = df[['MCID', 'Protein', 'Range', 'AA']]
# We so

print("DataFrame created successfully!")
print(f"Shape: {df.shape}")
print(f"\nColumn dtypes:")
print(df.dtypes)
print(f"\nDataFrame info:")
print(df.info())

DataFrame created successfully!
Shape: (1200826, 4)

Column dtypes:
MCID       object
Protein    object
Range      object
AA         object
dtype: object

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200826 entries, 0 to 1200825
Data columns (total 4 columns):
 #   Column   Non-Null Count    Dtype 
---  ------   --------------    ----- 
 0   MCID     1200826 non-null  object
 1   Protein  1200826 non-null  object
 2   Range    1200826 non-null  object
 3   AA       1200826 non-null  object
dtypes: object(4)
memory usage: 36.6+ MB
None


## Section 7: Display and Save Results

In [8]:
# Add a column for sequence length
df['AALength'] = df['AA'].apply(len)
# Rearrange  columns to put AA at the end
df = df[['MCID', 'Protein', 'Range', 'AALength', 'AA']]

In [9]:
# Display the head of the DataFrame
print("\nDataFrame head:")
df.head()


DataFrame head:


,MCID,Protein,Range,AALength,AA
0,MC113651,A0A0K6GCJ4,2-376,375,PTVVSSRALVPIGRRIRNAPEDSDASKAIVLRRKQQGETEDLSKAL...
1,MC113651,A8N621,40-384,345,TRALILRNGKHGAMGTGEIIASNKITGREKLDLLAEELIEKMKTAV...
2,MC113651,A0A0H2RRJ7,98-399,302,SNAVRAPFDLDSLIRWSQSQMDAALDQVNNLHETDFCYEIVERHIS...
3,MC113651,G4TJX7,26-369,344,VILSNQRREFVNTSSNAIALRYEPHGIWGEGQLASFKKRSGQEKLA...
4,MC113651,UPI0004449C2E,4-413,410,PPKDSKQPKDAPASTSRDLVRVGPRHVSAKHVVGRRNEDGQKPTTA...


In [10]:
# Summary statistics
print("\nSummary statistics:")
df.describe().T


Summary statistics:


,count,mean,std,min,25%,50%,75%,max
AALength,1200826.0,181.176998,181.632418,27.0,78.0,121.0,212.0,3545.0


In [11]:
# More derailed summary statistics
print("\n" + "="*80)
print("SUMMARY STATISTICS")
print("="*80)
print(f"Total rows: {len(df)}")
print(f"Total columns: {len(df.columns)}")
print(f"\nMCIDs represented: {df['MCID'].nunique()}")
print(f"Unique Proteins: {df['Protein'].nunique()}")

print(f"\nMCID distribution:")
print(df['MCID'].value_counts().head(10))

# Check for any missing values
print(f"\nMissing values:")
print(df.isnull().sum())

# Amino acid sequence length statistics
print(f"\nAmino acid sequence length statistics:")
df['AA_Length'] = df['AA'].str.len()
print(f"  Min length: {df['AA_Length'].min()}")
print(f"  Max length: {df['AA_Length'].max()}")
print(f"  Mean length: {df['AA_Length'].mean():.2f}")
print(f"  Median length: {df['AA_Length'].median():.2f}")


SUMMARY STATISTICS
Total rows: 1200826
Total columns: 5

MCIDs represented: 34556


Unique Proteins: 1014273

MCID distribution:
MCID
MC331055    49
MC343031    49
MC103575    49
MC30001     49
MC304682    49
MC88295     49
MC101962    49
MC14331     49
MC29941     49
MC102068    49
Name: count, dtype: int64

Missing values:
MCID        0
Protein     0
Range       0
AALength    0
AA          0
dtype: int64

Amino acid sequence length statistics:
  Min length: 27
  Max length: 3545
  Mean length: 181.18
  Median length: 121.00


In [12]:
# Info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200826 entries, 0 to 1200825
Data columns (total 6 columns):
 #   Column     Non-Null Count    Dtype 
---  ------     --------------    ----- 
 0   MCID       1200826 non-null  object
 1   Protein    1200826 non-null  object
 2   Range      1200826 non-null  object
 3   AALength   1200826 non-null  int64 
 4   AA         1200826 non-null  object
 5   AA_Length  1200826 non-null  int64 
dtypes: int64(2), object(4)
memory usage: 55.0+ MB


In [13]:
# Let's organise columns properly for PostgreSQL needs :
# 1. We drop AA_Length:
df = df.drop(columns=["AA_Length"])
# 2. We renamme columns properly:
df = df.rename(columns={
    "MCID":"mcid",
    "Protein":"protein_id",
    "Range":"seq_range",
    "AALength":"seq_length",
    "AA" : "aa_seq"
    })
df.head()

,mcid,protein_id,seq_range,seq_length,aa_seq
0,MC113651,A0A0K6GCJ4,2-376,375,PTVVSSRALVPIGRRIRNAPEDSDASKAIVLRRKQQGETEDLSKAL...
1,MC113651,A8N621,40-384,345,TRALILRNGKHGAMGTGEIIASNKITGREKLDLLAEELIEKMKTAV...
2,MC113651,A0A0H2RRJ7,98-399,302,SNAVRAPFDLDSLIRWSQSQMDAALDQVNNLHETDFCYEIVERHIS...
3,MC113651,G4TJX7,26-369,344,VILSNQRREFVNTSSNAIALRYEPHGIWGEGQLASFKKRSGQEKLA...
4,MC113651,UPI0004449C2E,4-413,410,PPKDSKQPKDAPASTSRDLVRVGPRHVSAKHVVGRRNEDGQKPTTA...


In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200826 entries, 0 to 1200825
Data columns (total 5 columns):
 #   Column      Non-Null Count    Dtype 
---  ------      --------------    ----- 
 0   mcid        1200826 non-null  object
 1   protein_id  1200826 non-null  object
 2   seq_range   1200826 non-null  object
 3   seq_length  1200826 non-null  int64 
 4   aa_seq      1200826 non-null  object
dtypes: int64(1), object(4)
memory usage: 45.8+ MB


In [15]:
# Save the DataFrame to CSV file
output_path = Path('/u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCFam/DPCFamB/dpcfamB_sequences.csv')
output_path.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(output_path, index=False)
print(f"\nDataFrame saved to: {output_path.absolute()}")
print(f"File size: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

print("\n" + "="*80)
print("PROCESSING COMPLETE!")
print("="*80)


DataFrame saved to: /u/mdmc/enyanduk/internship_areasciencepark/Dataframes/DPCFam/DPCFamB/dpcfamB_sequences.csv
File size: 242.41 MB

PROCESSING COMPLETE!
